# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.13 — FAST
## Linearized Vector Reduction and Static Scalar Poisson Gate Audit

`.3.3.12` a fermé le secteur tensoriel TT autour de Minkowski avec deux polarisations et :

\[
c_T^2=\frac{1}{1-c_1-c_3}.
\]

Il reste trois DOF non tensoriels.

Mission :

1. réduire explicitement le secteur vectoriel transverse ;
2. classifier sa dispersion et ses conditions de stabilité ;
3. effectuer le matching modal \(5=2_T+2_V+1_{\rm restant}\) si le secteur vectoriel passe ;
4. garder le secteur scalaire/Poisson verrouillé tant que l'action scalaire, les contraintes, le couplage matière et \(G_{\rm eff}\) ne sont pas réellement dérivés.

In [1]:
# VS13.1 — Environment and frozen upstream
from __future__ import annotations
import sympy as sp
import json, sys
from pathlib import Path

UPSTREAM = {
    "p3312": {
        "canonical_user_executed_sha256": "5103796a063c7be1f7becc2ce9c9dfc455fb43e7a86af7e16b2821b97a5c17ce",
        "canonical_user_executed_size_bytes": 23578,
        "SVT_DECOMPOSITION_MATERIALIZED": True,
        "TENSOR_DISPERSION_FULLY_CLASSIFIED": True,
        "TENSOR_LINEARIZED_BENCHMARK_PASS": True,
        "TENSOR_PROPAGATING_DOF": 2,
        "NON_TENSOR_DOF_REMAINDER": 3,
        "WEAK_FIELD_BENCHMARK_PASS": False,
        "SCHWARZSCHILD_BENCHMARK_AUTHORIZED": False,
    }
}
UPSTREAM_GATE = all([
    UPSTREAM["p3312"]["SVT_DECOMPOSITION_MATERIALIZED"],
    UPSTREAM["p3312"]["TENSOR_DISPERSION_FULLY_CLASSIFIED"],
    UPSTREAM["p3312"]["TENSOR_LINEARIZED_BENCHMARK_PASS"],
    UPSTREAM["p3312"]["TENSOR_PROPAGATING_DOF"] == 2,
    UPSTREAM["p3312"]["NON_TENSOR_DOF_REMAINDER"] == 3,
    not UPSTREAM["p3312"]["WEAK_FIELD_BENCHMARK_PASS"],
    not UPSTREAM["p3312"]["SCHWARZSCHILD_BENCHMARK_AUTHORIZED"],
])
assert UPSTREAM_GATE
print("Python =", sys.version.split()[0])
print("SymPy =", sp.__version__)
print("UPSTREAM_GATE =", UPSTREAM_GATE)
print("P3312_CANONICAL_SHA256 =", UPSTREAM["p3312"]["canonical_user_executed_sha256"])

Python = 3.12.13
SymPy = 1.14.0
UPSTREAM_GATE = True
P3312_CANONICAL_SHA256 = 5103796a063c7be1f7becc2ce9c9dfc455fb43e7a86af7e16b2821b97a5c17ce


# VS13.2 — Secteur vectoriel transverse

Pour une onde de nombre d'onde \(k\) selon \(z\), on choisit une polarisation transverse \(x\).

On note :

\[
W(t,z)
\]

la perturbation vectorielle GVH transverse,

\[
E(t,z)
\]

la perturbation métrique vectorielle transverse, et

\[
S(t,z)
\]

le shift transverse.

La combinaison entrant dans la courbure extrinsèque est :

\[
\kappa=\frac{k}{2}(\dot E-S).
\]

Dans ce sous-secteur :

\[
B_x=\dot W,
\]

\[
D_{zx}=kW+\kappa,
\qquad
D_{xz}=\kappa,
\qquad
\mathrm{tr}D=0.
\]

In [2]:
# VS13.3 — Exact transverse-vector quadratic form
c1,c2,c3,c4,k = sp.symbols("c1 c2 c3 c4 k", real=True)
Wdot,W,kap = sp.symbols("Wdot W kappa", real=True)

A_T = sp.factor(1-c1-c3)

L_EH_V = 2*kap**2
DijDij_V = sp.expand((k*W+kap)**2 + kap**2)
DijDji_V = sp.expand(2*kap*(k*W+kap))

L_GVH_V = sp.expand(
    (c1+c4)*Wdot**2
    - c1*DijDij_V
    - c3*DijDji_V
)
L_V = sp.expand(L_EH_V + L_GVH_V)

VECTOR_QUADRATIC_FORM_MATERIALIZED = True

print("L_EH_V =", L_EH_V)
print("L_GVH_V =", sp.factor(L_GVH_V))
print("L_V =", sp.factor(L_V))
print("VECTOR_QUADRATIC_FORM_MATERIALIZED =", VECTOR_QUADRATIC_FORM_MATERIALIZED)

L_EH_V = 2*kappa**2
L_GVH_V = -W**2*c1*k**2 - 2*W*c1*k*kappa - 2*W*c3*k*kappa + Wdot**2*c1 + Wdot**2*c4 - 2*c1*kappa**2 - 2*c3*kappa**2
L_V = -W**2*c1*k**2 - 2*W*c1*k*kappa - 2*W*c3*k*kappa + Wdot**2*c1 + Wdot**2*c4 - 2*c1*kappa**2 - 2*c3*kappa**2 + 2*kappa**2
VECTOR_QUADRATIC_FORM_MATERIALIZED = True


# VS13.4 — Contrainte de shift transverse

Le shift transverse est non dynamique.

On résout :

\[
\frac{\partial\mathcal L_V^{(2)}}{\partial\kappa}=0
\]

puis on réinjecte la solution dans l'action.

In [3]:
# VS13.5 — Solve and reduce shift constraint
dL_dkap = sp.factor(sp.diff(L_V,kap))
kap_sol = sp.factor(sp.solve(sp.Eq(dL_dkap,0), kap)[0])
L_V_red = sp.factor(sp.simplify(L_V.subs(kap,kap_sol)))

VECTOR_SHIFT_CONSTRAINT_SOLVED = True
VECTOR_REDUCED_ACTION_DERIVED = True

print("dL/dkappa =", dL_dkap)
print("kappa_solution =", kap_sol)
print("L_V_reduced =", L_V_red)
print("VECTOR_SHIFT_CONSTRAINT_SOLVED =", VECTOR_SHIFT_CONSTRAINT_SOLVED)
print("VECTOR_REDUCED_ACTION_DERIVED =", VECTOR_REDUCED_ACTION_DERIVED)

dL/dkappa = -2*(W*c1*k + W*c3*k + 2*c1*kappa + 2*c3*kappa - 2*kappa)
kappa_solution = -W*k*(c1 + c3)/(2*(c1 + c3 - 1))
L_V_reduced = -(W**2*c1**2*k**2 - 2*W**2*c1*k**2 - W**2*c3**2*k**2 - 2*Wdot**2*c1**2 - 2*Wdot**2*c1*c3 - 2*Wdot**2*c1*c4 + 2*Wdot**2*c1 - 2*Wdot**2*c3*c4 + 2*Wdot**2*c4)/(2*(c1 + c3 - 1))
VECTOR_SHIFT_CONSTRAINT_SOLVED = True
VECTOR_REDUCED_ACTION_DERIVED = True


# VS13.6 — Action vectorielle réduite et dispersion

On cherche :

\[
\mathcal L_V^{(2)}
=
K_V\dot W^2-G_Vk^2W^2.
\]

Le calcul donne :

\[
K_V=c_1+c_4,
\]

\[
G_V=
\frac{2c_1+c_3^2-c_1^2}{2(1-c_1-c_3)}.
\]

Donc :

\[
\boxed{
c_V^2=
\frac{2c_1+c_3^2-c_1^2}
{2(1-c_1-c_3)(c_1+c_4)}
}.
\]

In [4]:
# VS13.7 — Extract vector coefficients
K_V = sp.factor(c1+c4)
G_V = sp.factor((2*c1 + c3**2 - c1**2)/(2*(1-c1-c3)))
target = sp.expand(K_V*Wdot**2 - G_V*k**2*W**2)

VECTOR_REDUCED_ACTION_CROSSCHECK_PASS = (
    sp.simplify(sp.expand(L_V_red-target)) == 0
)

cV2 = sp.factor(G_V/K_V)

VECTOR_KINETIC_COEFFICIENT = K_V
VECTOR_GRADIENT_COEFFICIENT = G_V
VECTOR_SPEED_SQUARED = cV2
VECTOR_NO_GHOST_CONDITION = sp.StrictGreaterThan(K_V,0)
VECTOR_NO_GRADIENT_INSTABILITY_CONDITION = sp.StrictGreaterThan(G_V,0)
VECTOR_DISPERSION_CLASSIFIED = VECTOR_REDUCED_ACTION_CROSSCHECK_PASS

assert VECTOR_DISPERSION_CLASSIFIED

print("VECTOR_KINETIC_COEFFICIENT =", VECTOR_KINETIC_COEFFICIENT)
print("VECTOR_GRADIENT_COEFFICIENT =", VECTOR_GRADIENT_COEFFICIENT)
print("VECTOR_SPEED_SQUARED =", VECTOR_SPEED_SQUARED)
print("VECTOR_NO_GHOST_CONDITION =", VECTOR_NO_GHOST_CONDITION)
print("VECTOR_NO_GRADIENT_INSTABILITY_CONDITION =", VECTOR_NO_GRADIENT_INSTABILITY_CONDITION)
print("VECTOR_DISPERSION_CLASSIFIED =", VECTOR_DISPERSION_CLASSIFIED)

VECTOR_KINETIC_COEFFICIENT = c1 + c4
VECTOR_GRADIENT_COEFFICIENT = (c1**2 - 2*c1 - c3**2)/(2*(c1 + c3 - 1))
VECTOR_SPEED_SQUARED = (c1**2 - 2*c1 - c3**2)/(2*(c1 + c4)*(c1 + c3 - 1))
VECTOR_NO_GHOST_CONDITION = c1 + c4 > 0
VECTOR_NO_GRADIENT_INSTABILITY_CONDITION = (c1**2 - 2*c1 - c3**2)/(2*(c1 + c3 - 1)) > 0
VECTOR_DISPERSION_CLASSIFIED = True


# VS13.8 — Matching modal

Le secteur tensoriel fournit \(2\) DOF.

Le secteur vectoriel transverse possède deux polarisations :

\[
\boxed{2\ \text{DOF vectoriels}}.
\]

Avec le comptage canonique total :

\[
5,
\]

il reste :

\[
\boxed{1\ \text{DOF non classifié}}.
\]

On peut donc désormais écrire, au niveau du comptage :

\[
\boxed{
5=2_T+2_V+1_{\rm restant}
}
\]

sans encore identifier dynamiquement le dernier mode.

In [5]:
# VS13.9 — DOF matching
TOTAL_CANONICAL_DOF = 5
TENSOR_DOF = 2
VECTOR_DOF = 2
REMAINING_DOF = TOTAL_CANONICAL_DOF - TENSOR_DOF - VECTOR_DOF

VECTOR_PROPAGATING_DOF = 2
TENSOR_PLUS_VECTOR_DOF_MATCH_PASS = (
    TENSOR_DOF + VECTOR_DOF == 4 and REMAINING_DOF == 1
)

SCALAR_PROPAGATING_DOF_CLASSIFIED = False
FULL_LINEARIZED_DOF_MATCH_PASS = False

assert TENSOR_PLUS_VECTOR_DOF_MATCH_PASS
assert REMAINING_DOF == 1

print("TOTAL_CANONICAL_DOF =", TOTAL_CANONICAL_DOF)
print("TENSOR_DOF =", TENSOR_DOF)
print("VECTOR_DOF =", VECTOR_DOF)
print("REMAINING_DOF =", REMAINING_DOF)
print("TENSOR_PLUS_VECTOR_DOF_MATCH_PASS =", TENSOR_PLUS_VECTOR_DOF_MATCH_PASS)
print("FULL_LINEARIZED_DOF_MATCH_PASS =", FULL_LINEARIZED_DOF_MATCH_PASS)

TOTAL_CANONICAL_DOF = 5
TENSOR_DOF = 2
VECTOR_DOF = 2
REMAINING_DOF = 1
TENSOR_PLUS_VECTOR_DOF_MATCH_PASS = True
FULL_LINEARIZED_DOF_MATCH_PASS = False


# VS13.10 — Secteur scalaire statique / Poisson

Pour fermer le benchmark faible champ, il faut maintenant dériver :

\[
\nabla^2\Phi=4\pi G_{\rm eff}\rho.
\]

Cela exige encore :

1. l'action quadratique scalaire complète ;
2. les variables scalaires \(n,B,\psi,E,W\) ;
3. la réduction des contraintes de lapse et shift ;
4. la réduction seconde classe linéarisée ;
5. le couplage à la matière ;
6. l'identification du potentiel physique \(\Phi\) ;
7. le calcul de \(G_{\rm eff}\).

`.3.3.13` ne remplace aucun de ces éléments par une hypothèse.

In [6]:
# VS13.11 — Scalar/Poisson locks
SCALAR_QUADRATIC_ACTION_FULLY_DERIVED = False
SCALAR_LAPSE_SHIFT_CONSTRAINTS_SOLVED = False
SCALAR_SECOND_CLASS_REDUCTION_MATERIALIZED = False
MATTER_SOURCE_COUPLING_NORMALIZATION_MATERIALIZED = False
PHYSICAL_NEWTONIAN_POTENTIAL_IDENTIFIED = False
POISSON_EQUATION_DERIVED = False
G_EFFECTIVE_DERIVED = False
STATIC_WEAK_FIELD_LIMIT_CLASSIFIED = False

SCALAR_POISSON_SCOPE_DISCIPLINE_PASS = all([
    not SCALAR_QUADRATIC_ACTION_FULLY_DERIVED,
    not SCALAR_LAPSE_SHIFT_CONSTRAINTS_SOLVED,
    not SCALAR_SECOND_CLASS_REDUCTION_MATERIALIZED,
    not MATTER_SOURCE_COUPLING_NORMALIZATION_MATERIALIZED,
    not PHYSICAL_NEWTONIAN_POTENTIAL_IDENTIFIED,
    not POISSON_EQUATION_DERIVED,
    not G_EFFECTIVE_DERIVED,
    not STATIC_WEAK_FIELD_LIMIT_CLASSIFIED,
])
assert SCALAR_POISSON_SCOPE_DISCIPLINE_PASS

print("SCALAR_QUADRATIC_ACTION_FULLY_DERIVED =", SCALAR_QUADRATIC_ACTION_FULLY_DERIVED)
print("POISSON_EQUATION_DERIVED =", POISSON_EQUATION_DERIVED)
print("G_EFFECTIVE_DERIVED =", G_EFFECTIVE_DERIVED)
print("STATIC_WEAK_FIELD_LIMIT_CLASSIFIED =", STATIC_WEAK_FIELD_LIMIT_CLASSIFIED)

SCALAR_QUADRATIC_ACTION_FULLY_DERIVED = False
POISSON_EQUATION_DERIVED = False
G_EFFECTIVE_DERIVED = False
STATIC_WEAK_FIELD_LIMIT_CLASSIFIED = False


# VS13.12 — Région stable tensorielle + vectorielle

Conditions déjà classifiées :

\[
1-c_1-c_3>0,
\]

\[
c_1+c_4>0,
\]

\[
\frac{2c_1+c_3^2-c_1^2}{2(1-c_1-c_3)}>0.
\]

Ces conditions concernent uniquement les sous-secteurs déjà réduits.

In [7]:
# VS13.13 — Nonempty stable tensor+vector witness
witness = {
    c1: sp.Rational(1,10),
    c3: sp.Rational(1,5),
    c4: sp.Rational(1,2),
}
KT_w = sp.simplify((1-c1-c3).subs(witness))
KV_w = sp.simplify(K_V.subs(witness))
GV_w = sp.simplify(G_V.subs(witness))
cV2_w = sp.simplify(cV2.subs(witness))

TENSOR_VECTOR_STABLE_REGION_NONEMPTY = all([
    KT_w > 0,
    KV_w > 0,
    GV_w > 0,
    cV2_w > 0,
])
assert TENSOR_VECTOR_STABLE_REGION_NONEMPTY

print("witness =", witness)
print("K_T(witness) =", KT_w)
print("K_V(witness) =", KV_w)
print("G_V(witness) =", GV_w)
print("c_V^2(witness) =", cV2_w)
print("TENSOR_VECTOR_STABLE_REGION_NONEMPTY =", TENSOR_VECTOR_STABLE_REGION_NONEMPTY)

witness = {c1: 1/10, c3: 1/5, c4: 1/2}
K_T(witness) = 7/10
K_V(witness) = 3/5
G_V(witness) = 23/140
c_V^2(witness) = 23/84
TENSOR_VECTOR_STABLE_REGION_NONEMPTY = True


# VS13.14 — Verdict

Le secteur vectoriel transverse est fermé dans le scope du notebook.

Le faible champ global reste `PARTIAL`, car le dernier DOF et la limite de Poisson restent ouverts.

In [8]:
# VS13.15 — Final classifier
VECTOR_LINEARIZED_BENCHMARK_PASS = all([
    VECTOR_QUADRATIC_FORM_MATERIALIZED,
    VECTOR_SHIFT_CONSTRAINT_SOLVED,
    VECTOR_REDUCED_ACTION_DERIVED,
    VECTOR_REDUCED_ACTION_CROSSCHECK_PASS,
    VECTOR_DISPERSION_CLASSIFIED,
    VECTOR_PROPAGATING_DOF == 2,
    TENSOR_VECTOR_STABLE_REGION_NONEMPTY,
])

WEAK_FIELD_BENCHMARK_PASS = all([
    VECTOR_LINEARIZED_BENCHMARK_PASS,
    SCALAR_PROPAGATING_DOF_CLASSIFIED,
    FULL_LINEARIZED_DOF_MATCH_PASS,
    STATIC_WEAK_FIELD_LIMIT_CLASSIFIED,
    POISSON_EQUATION_DERIVED,
    G_EFFECTIVE_DERIVED,
])

WEAK_FIELD_BENCHMARK_STATUS = (
    "PASS" if WEAK_FIELD_BENCHMARK_PASS
    else "PARTIAL_PASS_TENSOR_AND_VECTOR_CLASSIFIED_SCALAR_POISSON_PENDING"
)

SCHWARZSCHILD_BENCHMARK_AUTHORIZED = WEAK_FIELD_BENCHMARK_PASS
CLASSICAL_PREDICTIONS_AUTHORIZED = False
QUANTIZATION_READY = False

VS13_OBSTRUCTIONS = []
if not SCALAR_QUADRATIC_ACTION_FULLY_DERIVED:
    VS13_OBSTRUCTIONS.append("DERIVE-FULL-LINEARIZED-SCALAR-QUADRATIC-ACTION")
if not SCALAR_LAPSE_SHIFT_CONSTRAINTS_SOLVED:
    VS13_OBSTRUCTIONS.append("SOLVE-SCALAR-LAPSE-SHIFT-CONSTRAINTS")
if not SCALAR_SECOND_CLASS_REDUCTION_MATERIALIZED:
    VS13_OBSTRUCTIONS.append("MATERIALIZE-SCALAR-SECOND-CLASS-REDUCTION")
if not SCALAR_PROPAGATING_DOF_CLASSIFIED:
    VS13_OBSTRUCTIONS.append("CLASSIFY-REMAINING-SCALAR-DOF")
if not POISSON_EQUATION_DERIVED:
    VS13_OBSTRUCTIONS.append("DERIVE-POISSON-EQUATION")
if not G_EFFECTIVE_DERIVED:
    VS13_OBSTRUCTIONS.append("DERIVE-G_EFFECTIVE")

VS13_LOCAL_AUDIT_PASS = all([
    VECTOR_LINEARIZED_BENCHMARK_PASS,
    not WEAK_FIELD_BENCHMARK_PASS,
    not SCHWARZSCHILD_BENCHMARK_AUTHORIZED,
    not CLASSICAL_PREDICTIONS_AUTHORIZED,
    not QUANTIZATION_READY,
    len(VS13_OBSTRUCTIONS) > 0,
])

VS13_NEXT_AUTHORIZED = (
    "AUDIT-LINEARIZED-SCALAR-REDUCTION-POISSON-AND-G_EFFECTIVE"
    if VS13_LOCAL_AUDIT_PASS
    else "REPAIR-.3.3.13-VECTOR-REDUCTION"
)

assert VS13_LOCAL_AUDIT_PASS

print("VECTOR_LINEARIZED_BENCHMARK_PASS =", VECTOR_LINEARIZED_BENCHMARK_PASS)
print("VECTOR_SPEED_SQUARED =", VECTOR_SPEED_SQUARED)
print("REMAINING_DOF =", REMAINING_DOF)
print("WEAK_FIELD_BENCHMARK_PASS =", WEAK_FIELD_BENCHMARK_PASS)
print("WEAK_FIELD_BENCHMARK_STATUS =", WEAK_FIELD_BENCHMARK_STATUS)
print("SCHWARZSCHILD_BENCHMARK_AUTHORIZED =", SCHWARZSCHILD_BENCHMARK_AUTHORIZED)
print("VS13_OBSTRUCTIONS =", VS13_OBSTRUCTIONS)
print("VS13_NEXT_AUTHORIZED =", VS13_NEXT_AUTHORIZED)

VECTOR_LINEARIZED_BENCHMARK_PASS = True
VECTOR_SPEED_SQUARED = (c1**2 - 2*c1 - c3**2)/(2*(c1 + c4)*(c1 + c3 - 1))
REMAINING_DOF = 1
WEAK_FIELD_BENCHMARK_PASS = False
WEAK_FIELD_BENCHMARK_STATUS = PARTIAL_PASS_TENSOR_AND_VECTOR_CLASSIFIED_SCALAR_POISSON_PENDING
SCHWARZSCHILD_BENCHMARK_AUTHORIZED = False
VS13_OBSTRUCTIONS = ['DERIVE-FULL-LINEARIZED-SCALAR-QUADRATIC-ACTION', 'SOLVE-SCALAR-LAPSE-SHIFT-CONSTRAINTS', 'MATERIALIZE-SCALAR-SECOND-CLASS-REDUCTION', 'CLASSIFY-REMAINING-SCALAR-DOF', 'DERIVE-POISSON-EQUATION', 'DERIVE-G_EFFECTIVE']
VS13_NEXT_AUTHORIZED = AUDIT-LINEARIZED-SCALAR-REDUCTION-POISSON-AND-G_EFFECTIVE


# VS13.16 — Portée exacte

Un PASS local de `.3.3.13` signifie :

\[
\boxed{
2_T+2_V
}
\]

sont explicitement classifiés autour de Minkowski.

Il reste :

\[
\boxed{1\ \text{DOF}}
\]

à identifier dans le secteur scalaire.

Le benchmark faible champ global ne peut être déclaré PASS avant :

\[
\boxed{
\text{réduction scalaire}
+\text{Poisson}
+G_{\rm eff}.
}
\]

In [9]:
# VS13.17 — Artifact JSON
artifact = {
    "notebook": "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.13_Linearized_Vector_Reduction_and_Static_Scalar_Poisson_Gate_Audit_FAST",
    "execution_scope": "LINEARIZED_VECTOR_REDUCTION_AND_STATIC_SCALAR_POISSON_GATE_AROUND_MINKOWSKI",
    "upstream": UPSTREAM,
    "vector_sector": {
        "quadratic_form_materialized": VECTOR_QUADRATIC_FORM_MATERIALIZED,
        "shift_constraint_solved": VECTOR_SHIFT_CONSTRAINT_SOLVED,
        "reduced_action_derived": VECTOR_REDUCED_ACTION_DERIVED,
        "reduced_action_crosscheck_pass": VECTOR_REDUCED_ACTION_CROSSCHECK_PASS,
        "kinetic_coefficient": str(VECTOR_KINETIC_COEFFICIENT),
        "gradient_coefficient": str(VECTOR_GRADIENT_COEFFICIENT),
        "speed_squared": str(VECTOR_SPEED_SQUARED),
        "no_ghost_condition": str(VECTOR_NO_GHOST_CONDITION),
        "no_gradient_instability_condition": str(VECTOR_NO_GRADIENT_INSTABILITY_CONDITION),
        "dispersion_classified": VECTOR_DISPERSION_CLASSIFIED,
        "propagating_dof": VECTOR_PROPAGATING_DOF,
    },
    "dof": {
        "total": TOTAL_CANONICAL_DOF,
        "tensor": TENSOR_DOF,
        "vector": VECTOR_DOF,
        "remaining": REMAINING_DOF,
        "full_linearized_match_pass": FULL_LINEARIZED_DOF_MATCH_PASS,
    },
    "scalar_static": {
        "quadratic_action_fully_derived": SCALAR_QUADRATIC_ACTION_FULLY_DERIVED,
        "lapse_shift_constraints_solved": SCALAR_LAPSE_SHIFT_CONSTRAINTS_SOLVED,
        "second_class_reduction_materialized": SCALAR_SECOND_CLASS_REDUCTION_MATERIALIZED,
        "scalar_dof_classified": SCALAR_PROPAGATING_DOF_CLASSIFIED,
        "poisson_equation_derived": POISSON_EQUATION_DERIVED,
        "G_effective_derived": G_EFFECTIVE_DERIVED,
        "static_weak_field_limit_classified": STATIC_WEAK_FIELD_LIMIT_CLASSIFIED,
    },
    "scientific_status": {
        "VECTOR_LINEARIZED_BENCHMARK_PASS": VECTOR_LINEARIZED_BENCHMARK_PASS,
        "WEAK_FIELD_BENCHMARK_PASS": WEAK_FIELD_BENCHMARK_PASS,
        "WEAK_FIELD_BENCHMARK_STATUS": WEAK_FIELD_BENCHMARK_STATUS,
        "SCHWARZSCHILD_BENCHMARK_AUTHORIZED": SCHWARZSCHILD_BENCHMARK_AUTHORIZED,
        "CLASSICAL_PREDICTIONS_AUTHORIZED": CLASSICAL_PREDICTIONS_AUTHORIZED,
        "QUANTIZATION_READY": QUANTIZATION_READY,
    },
    "verdict": {
        "VS13_LOCAL_AUDIT_PASS": VS13_LOCAL_AUDIT_PASS,
        "obstructions": VS13_OBSTRUCTIONS,
    },
    "next_authorized": VS13_NEXT_AUTHORIZED,
    "scope_note": "Tensor and transverse-vector sectors classified around Minkowski; scalar reduction and Poisson/G_eff remain open."
}

export_dir = Path("/content/gvh_exports") if Path("/content").exists() else Path("/mnt/data")
export_dir.mkdir(parents=True, exist_ok=True)
artifact_path = export_dir / "gvh_0.3.2.7.3.7.3.3.13_Linearized_Vector_Reduction_and_Static_Scalar_Poisson_Gate_Audit_FAST.json"
artifact_path.write_text(json.dumps(artifact, indent=2, ensure_ascii=False), encoding="utf-8")
print("VS13 artifact =", artifact_path)

VS13 artifact = /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.13_Linearized_Vector_Reduction_and_Static_Scalar_Poisson_Gate_Audit_FAST.json


# Conclusion

`.3.3.13 FAST` ferme le secteur vectoriel transverse et laisse le secteur scalaire/Poisson comme dernier verrou faible champ.

Prochaine étape autorisée :

\[
\boxed{
\texttt{AUDIT-LINEARIZED-SCALAR-REDUCTION-POISSON-AND-G\_EFFECTIVE}.
}
\]